## Working with annotation volumes

Annotation volumes are useful references for understanding brain parcellations at multiple levels. For example, a cell's location can be described as being in the lateral geniculate nucleus or in any of its parent structures in the anatomical ontology (dorsal thalamus, thalamus, diencephalon, etc.). Annotation volumes can label atlas-registered data with terms from the atlas ontology.

This tutorial shows some examples for using the Allen adult mouse brain annotation volume (`allen_mouse_10um`) and the [BrainGlobe Atlas API](https://brainglobe.info/documentation/brainglobe-atlasapi/api.html) to (1) query labels and (2) extract regions at any level as a mask for reference and visualization.

In [ ]:
import numpy as np
import pandas as pd
from brainglobe_atlasapi import BrainGlobeAtlas
from brainglobe_space import AnatomicalSpace

First, we will load the atlas from the local BrainGlobe cache. In this tutorial, we will assume that you have already downloaded the atlas (see [Getting started](./getting_started_brainglobe.ipynb)); otherwise it is downloaded on first use.

The annotation image is an array where each voxel is assigned a numeric `annotation value` that corresponds to the most granular structure that is annotated in the volume. To convert the numeric value to a label, we use the atlas structure table (`lookup_df`).

In [ ]:
atlas = BrainGlobeAtlas("allen_mouse_10um")
annotation_image = atlas.annotation

print(f"Annotation image has dimensions: {annotation_image.shape}")
print(f"Orientation: {atlas.orientation}, resolution (um): {atlas.resolution}")

The structure table contains the annotation label value (`id`) as well as the label name and acronym.

In [ ]:
atlas.lookup_df.head(10)

#### 1. Query annotation labels from coordinates

Now that we have loaded in the atlas, we can use it to query (x,y,z) coordinates and get the anatomical structure label for that coordinate voxel. Coordinates can be represented either by the array index or the physical space. The array index follows the atlas orientation (`asr`: axes are anterior-posterior, superior-inferior, and right-left), and `AnatomicalSpace` describes how that space is laid out. Physical coordinates (in micrometers) are the index multiplied by the resolution.

Let's use the index `[1049, 419, 690]` as a reference

![GPe example](../images/human_GPe_coordinate.png)

In [ ]:
space = AnatomicalSpace(atlas.orientation, shape=atlas.shape, resolution=atlas.resolution)
print(f"Atlas space: origin={space.origin}, shape={space.shape}, resolution={space.resolution}")

query_coordinate = [1049, 419, 690]
acronym = atlas.structure_from_coords(query_coordinate, as_acronym=True)
name = atlas.structures[acronym]["name"]

print(f"Coordinate {query_coordinate} is in {acronym}: {name}")

# Physical coordinates: pending an update to the API
# query_coordinate_physical = [121, -120, 530]
# acronym = atlas.structure_from_coords(query_coordinate_physical, microns=True, as_acronym=True)
# name = atlas.structures[acronym]["name"]
# print(f"Coordinate {query_coordinate_physical} is in {acronym}: {name}")

##### 1.1 Assign annotation labels to existing data table

Often, we will have sets of coordinates that define points/landmarks of interest (eg. soma locations). Rather than labeling them individually, we can annotate the input data table with the appropriate anatomical labels.

Here, we will use a short example of randomly sampled points within the annotated brain volume. In this example, the coordinates are indices rather than in physical dimensions.

In [ ]:
rng = np.random.default_rng(0)  # fixed seed for reproducibility

points = []
while len(points) < 5:
    index = tuple(int(rng.integers(0, s)) for s in atlas.shape)
    if annotation_image[index] > 0:  # keep only points inside the annotated brain
        points.append(index)

input_data = pd.DataFrame(points, columns=["x", "y", "z"])
input_data.head(5)

In [ ]:
labeled_data = input_data.copy()
labeled_data["acronym"] = [atlas.structure_from_coords(tuple(row), as_acronym=True) for row in input_data[["x", "y", "z"]].to_numpy()]
labeled_data["name"] = labeled_data["acronym"].map(lambda a: atlas.structures[a]["name"])
labeled_data.head(5)

#### 2. Create region masks

Annotation volumes can also be used to generate region masks for visualization or as a reference. For example, we may need to create a volume for the parabrachial nucleus for downstream use in visualizing region boundaries. However, the annotation volumes only show the most granular structures. To create a volume for the parabrachial nucleus, we will need to combine and aggregate all structures that are part of it.

![Dorsal striatum children structures](../images/dorsal_striatum_mesh_children.png)

This can be done using the `BrainGlobeAtlas` object we created earlier and the **structure acronym**. Here the mask is built in memory and is not saved to disk.

In [ ]:
region_mask = atlas.get_structure_mask('PB')
region_voxels = int((region_mask > 0).sum())
voxel_volume_mm3 = np.prod(atlas.resolution) / 1e9

print(f"PB mask has dimensions {region_mask.shape} and contains {region_voxels} voxels ({region_voxels * voxel_volume_mm3:.3f} mm^3)")

![Dorsal striatum mask](../images/dorsal_striatum_mesh.png)

This creates a volume that can be used for masking and visualization. The mask has the same shape as the annotation volume, so it can be used to index it.

##### 2.1 Get descendant structure names

Related to the above example, we may need to get a list of descendant structures associated with a region, eg. which labeled structures are part of `PB`?

The atlas object can be used to query a structure and return its descendants. To convert the acronyms into full names, we look them up in `atlas.structures`.

In [ ]:
query_structure = 'PB'
descendant_acronym = atlas.get_structure_descendants(query_structure)
descendant_name = [atlas.structures[a]["name"] for a in descendant_acronym]
print(f"Descendants of {query_structure}: {descendant_acronym}")
print(f"Descendants of {query_structure}: {descendant_name}")